In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

RAW_DATA_DIR = "/kaggle/input/datasets/dhoogla/cicids2017"
WORK_DIR = "/kaggle/working"

PARQUET_FILES = [
    "Benign-Monday-no-metadata.parquet",
    "Botnet-Friday-no-metadata.parquet",
    "Bruteforce-Tuesday-no-metadata.parquet",
    "DDoS-Friday-no-metadata.parquet",
    "DoS-Wednesday-no-metadata.parquet",
    "Infiltration-Thursday-no-metadata.parquet",
    "Portscan-Friday-no-metadata.parquet",
    "WebAttacks-Thursday-no-metadata.parquet",
]

In [ ]:
def infer_attack_category_from_filename(fname: str) -> str:
    base = os.path.basename(fname).lower()
    if "benign" in base:
        return "Normal"
    if "botnet" in base:
        return "Bot"
    if "bruteforce" in base:
        return "BruteForce"
    if "ddos" in base:
        return "DDoS"
    if "dos" in base:
        return "DoS"
    if "portscan" in base:
        return "PortScan"
    if "webattacks" in base:
        return "WebAttack"
    if "infiltration" in base:
        return "Infiltration"
    return "Unknown"

In [ ]:
dfs = []
for fname in PARQUET_FILES:
    fpath = os.path.join(RAW_DATA_DIR, fname)
    df_part = pd.read_parquet(fpath)
    df_part["attack_category"] = infer_attack_category_from_filename(fname)
    dfs.append(df_part)

df_raw = pd.concat(dfs, ignore_index=True)

print("Raw shape:", df_raw.shape)
df_raw.head()

In [ ]:
print("Raw DataFrame info:")
df_raw.info()

raw_memory_mb = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nApprox. raw memory usage: {raw_memory_mb:.2f} MB")

raw_missing = df_raw.isna().sum()
raw_missing[raw_missing > 0].sort_values(ascending=False)

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
df_raw[numeric_cols].describe().T.head()

In [ ]:
before_duplicates = len(df_raw)
df_no_dup = df_raw.drop_duplicates()
after_duplicates = len(df_no_dup)

print(f"Removed {before_duplicates - after_duplicates} duplicate rows "
      f"({(before_duplicates - after_duplicates) / before_duplicates:.4%})")

In [ ]:
df_clean = df_no_dup.copy()

# Count NaN before
nan_before = df_clean.isna().sum().sum()

# Replace inf / -inf with NaN, then drop rows with any NaN
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
nan_after_inf = df_clean.isna().sum().sum()

print(f"Total NaNs before replacing inf: {nan_before}")
print(f"Total NaNs after marking inf as NaN: {nan_after_inf}")

rows_before_nan_drop = len(df_clean)
df_clean = df_clean.dropna(axis=0)
rows_after_nan_drop = len(df_clean)

print(f"Dropped {rows_before_nan_drop - rows_after_nan_drop} rows with NaN "
      f"({(rows_before_nan_drop - rows_after_nan_drop) / rows_before_nan_drop:.4%})")

In [ ]:
# Optional: restrict inf/NaN cleaning to numeric columns
# df_clean[numeric_cols] = df_clean[numeric_cols].replace([np.inf, -np.inf], np.nan)
# df_clean.dropna(subset=numeric_cols, inplace=True)

In [ ]:
# Example sanity checks
print("Min/Max of Flow Duration:", df_clean["Flow Duration"].min(), df_clean["Flow Duration"].max())
print("Min/Max Flow Bytes/s:", df_clean["Flow Bytes/s"].min(), df_clean["Flow Bytes/s"].max())
print("Min/Max Flow Packets/s:", df_clean["Flow Packets/s"].min(), df_clean["Flow Packets/s"].max())

In [ ]:
# Example clipping for clearly invalid durations (if present)
rows_before_clip = len(df_clean)
invalid_duration_mask = df_clean["Flow Duration"] <= 0

print("Rows with non-positive Flow Duration:", invalid_duration_mask.sum())

# If you want to drop them:
df_clean = df_clean[~invalid_duration_mask].copy()
rows_after_clip = len(df_clean)

print(f"Dropped {rows_before_clip - rows_after_clip} rows with non-positive Flow Duration")

In [ ]:
# Sample for unsupervised outlier analysis
IF_SAMPLE_SIZE = 200_000

if len(df_clean) > IF_SAMPLE_SIZE:
    df_if = df_clean.sample(IF_SAMPLE_SIZE, random_state=42)
else:
    df_if = df_clean.copy()

numeric_cols = df_if.select_dtypes(include=[np.number]).columns

X_if = df_if[numeric_cols].astype(np.float32)

iso = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

iso.fit(X_if)
scores = iso.decision_function(X_if)
preds = iso.predict(X_if)  # -1 = outlier, 1 = inlier

df_if["if_score"] = scores
df_if["if_label"] = preds

df_if["if_label"].value_counts(normalize=True)

In [ ]:
outlier_rate_by_attack = (
    df_if.groupby(["attack_category", "if_label"])
         .size()
         .unstack(fill_value=0)
)

outlier_rate_by_attack["outlier_ratio"] = (
    outlier_rate_by_attack.get(-1, 0) /
    outlier_rate_by_attack.sum(axis=1)
)

outlier_rate_by_attack

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_if["if_score"], bins=50, kde=True)
plt.title("Isolation Forest anomaly scores (sample)")
plt.xlabel("Score (higher = more normal)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("isolation_forest_scores_hist.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def summarize_df(df: pd.DataFrame, name: str) -> pd.Series:
    n_rows = len(df)
    n_cols = df.shape[1]
    memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    n_duplicates = df.duplicated().sum()
    n_missing = df.isna().sum().sum()
    return pd.Series(
        {
            "rows": n_rows,
            "columns": n_cols,
            "memory_mb": memory_mb,
            "duplicate_rows": n_duplicates,
            "total_missing_values": n_missing,
        },
        name=name,
    )

summary_before = summarize_df(df_raw, "Before cleaning")
summary_after = summarize_df(df_clean, "After cleaning")

cleaning_summary = pd.concat([summary_before, summary_after], axis=1)
cleaning_summary

In [ ]:
cleaning_summary.to_csv("data_cleaning_summary_table1.csv")
cleaning_summary

In [ ]:
output_path = os.path.join(WORK_DIR, "cleaned_dataset.parquet")
df_clean.to_parquet(output_path, index=False)
print("Saved cleaned dataset to:", output_path)

In [ ]:
print("Class distribution before cleaning:")
print(df_raw["attack_category"].value_counts(normalize=True).mul(100).round(2))

print("\nClass distribution after cleaning:")
print(df_clean["attack_category"].value_counts(normalize=True).mul(100).round(2))